# Predicting F1 race points with Random Forest and MLflow

In [0]:
import mlflow
print(mlflow.__version__)

In [0]:
spark.sql("USE CATALOG gr5069")
spark.sql("USE SCHEMA jz4007")
spark.sql("SHOW TABLES").show(truncate=False)

In [0]:
from pyspark.sql import functions as F

results = spark.table("results")
drivers = spark.table("drivers")
races = spark.table("races")

results.printSchema()

In [0]:
model_df = (
    results.join(races.select("raceId", "year", "round"), on="raceId", how="left")
    .select(
        "points",
        "grid",
        "driverId",
        "constructorId",
        "year",
        "round"
    )
    .dropna()
)

print(model_df.count())
display(model_df)

In [0]:
pdf = model_df.toPandas()

X = pdf[["grid", "driverId", "constructorId", "year", "round"]]
y = pdf["points"]

print(X.shape)
print(y.shape)
X.head()

In [0]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

In [0]:
rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("MSE:", mse)
print("MAE:", mae)
print("R2:", r2)

In [0]:
import os
import tempfile
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn

mlflow.set_experiment("/Users/jz4007@columbia.edu/homework3_rf_points")

with mlflow.start_run(run_name="rf_baseline"):

    mlflow.log_param("model_type", "RandomForestRegressor")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 10)
    mlflow.log_param("random_state", 42)
    mlflow.log_param("test_size", 0.2)

    mlflow.log_metric("mse", mse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)

    mlflow.sklearn.log_model(rf, "random_forest_model")

    tmp_dir = tempfile.mkdtemp()

    pred_df = pd.DataFrame({
        "actual": y_test.values,
        "predicted": y_pred
    })
    csv_path = os.path.join(tmp_dir, "predictions.csv")
    pred_df.to_csv(csv_path, index=False)
    mlflow.log_artifact(csv_path)

    plt.figure(figsize=(6, 4))
    plt.scatter(y_test, y_pred, alpha=0.5)
    plt.xlabel("Actual Points")
    plt.ylabel("Predicted Points")
    plt.title("Actual vs Predicted")
    plot_path = os.path.join(tmp_dir, "actual_vs_predicted.png")
    plt.savefig(plot_path, bbox_inches="tight")
    plt.close()
    mlflow.log_artifact(plot_path)

print("logged to MLflow")

In [0]:
mlflow.set_experiment("/Users/jz4007@columbia.edu/homework3_rf_points")

In [0]:
import os
import tempfile
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn

param_grid = [
    {"n_estimators": 50, "max_depth": 5},
    {"n_estimators": 50, "max_depth": 10},
    {"n_estimators": 50, "max_depth": 15},
    {"n_estimators": 100, "max_depth": 5},
    {"n_estimators": 100, "max_depth": 10},
    {"n_estimators": 100, "max_depth": 15},
    {"n_estimators": 150, "max_depth": 5},
    {"n_estimators": 150, "max_depth": 10},
    {"n_estimators": 150, "max_depth": 15},
    {"n_estimators": 200, "max_depth": 10},
]

for i, params in enumerate(param_grid, start=1):
    rf = RandomForestRegressor(
        n_estimators=params["n_estimators"],
        max_depth=params["max_depth"],
        random_state=42
    )

    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)

    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    with mlflow.start_run(run_name=f"rf_full_run_{i}"):
        mlflow.log_param("model_type", "RandomForestRegressor")
        mlflow.log_param("n_estimators", params["n_estimators"])
        mlflow.log_param("max_depth", params["max_depth"])
        mlflow.log_param("random_state", 42)
        mlflow.log_param("test_size", 0.2)

        mlflow.log_metric("mse", mse)
        mlflow.log_metric("mae", mae)
        mlflow.log_metric("r2", r2)

        mlflow.sklearn.log_model(rf, "random_forest_model")

        tmp_dir = tempfile.mkdtemp()

        pred_df = pd.DataFrame({
            "actual": y_test.values,
            "predicted": y_pred
        })
        pred_path = os.path.join(tmp_dir, f"predictions_run_{i}.csv")
        pred_df.to_csv(pred_path, index=False)
        mlflow.log_artifact(pred_path)

        fi_df = pd.DataFrame({
            "feature": X_train.columns,
            "importance": rf.feature_importances_
        }).sort_values("importance", ascending=False)
        fi_path = os.path.join(tmp_dir, f"feature_importance_run_{i}.csv")
        fi_df.to_csv(fi_path, index=False)
        mlflow.log_artifact(fi_path)

        plt.figure(figsize=(6, 4))
        plt.scatter(y_test, y_pred, alpha=0.4)
        plt.xlabel("Actual Points")
        plt.ylabel("Predicted Points")
        plt.title(f"Actual vs Predicted Run {i}")
        plot_path = os.path.join(tmp_dir, f"actual_vs_predicted_run_{i}.png")
        plt.savefig(plot_path, bbox_inches="tight")
        plt.close()
        mlflow.log_artifact(plot_path)

print("10 full runs completed")

In [0]:
import mlflow

experiment = mlflow.get_experiment_by_name("/Users/jz4007@columbia.edu/homework3_rf_points")
runs_df = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

runs_df[["run_id", "tags.mlflow.runName", "metrics.r2", "metrics.mse", "metrics.mae", "params.n_estimators", "params.max_depth"]].sort_values(
    by="metrics.r2", ascending=False
).head(10)

In [0]:
best_run = runs_df.sort_values(by="metrics.r2", ascending=False).iloc[0]

print("Best run name:", best_run["tags.mlflow.runName"])
print("Best R2:", best_run["metrics.r2"])
print("Best MSE:", best_run["metrics.mse"])
print("Best MAE:", best_run["metrics.mae"])
print("n_estimators:", best_run["params.n_estimators"])
print("max_depth:", best_run["params.max_depth"])


## MLflow Homepage
![MLflow Homepage](IMAGE/MLFLOW_HOMEPAGE.png)

## Detailed Run Pages
![Run 1](IMAGE/RUN1PAGE.png)
![Run 2](IMAGE/RUN2PAGE.png)
![Run 3](IMAGE/RUN3PAGE.png)
![Run 4](IMAGE/RUN4PAGE.png)
![Run 5](IMAGE/RUN5PAGE.png)
![Run 6](IMAGE/RUN6PAGE.png)
![Run 7](IMAGE/RUN7PAGE.png)
![Run 8](IMAGE/RUN8PAGE.png)
![Run 9](IMAGE/RUN9PAGE.png)
![Run 10](IMAGE/RUN10PAGE.png)

I selected **rf_full_run_10** as the best model run. This run achieved the highest **R²** among all recorded runs, with an R² of approximately **0.5925**. It also had the lowest **MSE** (about **7.8778**) and the lowest **MAE** (about **1.5639**) compared with the other runs I tested. The model used **200 trees** (`n_estimators=200`) and a **maximum depth of 10** (`max_depth=10`).

I selected this run as the best model because it provided the strongest overall predictive performance on the test set. Since this is a regression problem, a higher R² indicates that the model explains more of the variation in race points, while lower MSE and MAE indicate smaller prediction errors. Based on these three metrics together, `rf_full_run_10` was the strongest model in my experiment.